In [ ]:
# Setup: run this cell first (click it, then press Shift+Enter).
# It finds the lab folder so the imports below work, wherever you opened this notebook from.
import sys
from pathlib import Path
if "google.colab" in sys.modules and not any((p / "labkit").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    # In Colab, fetch a copy of the repo first so the toolkit and data are available.
    import os, subprocess
    subprocess.run(["git", "clone", "-q", "https://github.com/andrewcberkley/models-make-you-lose-control.git"], check=True)
    os.chdir("models-make-you-lose-control/notebooks")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "labkit").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import math
import numpy as np
import pandas as pd
from labkit.checks import check, check_value

print("Ready.")

Ready.


# Chapter 1. How sure can you be when nothing went wrong?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewcberkley/models-make-you-lose-control/blob/main/solutions/ch01_zero_failures_solution.ipynb)

## The situation

You're embedded at **This Is Not a Test Labs**, and the team tells you, *"Our red team tried 300 times to get the agent to copy data out without permission, and the monitor caught every attempt. So the miss rate is zero."*

Before you accept that or push back, you need to know what 300 clean runs can and can't tell you, and that's what this chapter is about.

> **Why the name?** I named it after Missy Elliott's 2003 album *This Is Not a Test!*, which felt fitting because 300 clean runs are still only tests (and a model that can tell a test from the real thing might behave differently once it decides this is not a test).

## By the end you'll be able to

- explain why zero failures in a test doesn't mean a zero failure rate
- compute the highest failure rate that's still consistent with seeing zero failures (i.e., the 95% upper bound), both exactly and with a shortcut called the rule of three
- work out how many clean runs a stronger claim would need
- say the result in one clear, honest sentence

## Why it matters

A lot of safety evidence takes the form "we tried it many times and nothing bad happened," and this chapter shows you how to weigh that kind of evidence. You'll use it in the Chapter 1 threat-model exercise (to say how much testing a claim needs), and again in later chapters on safety cases and monitors.

## Part 1. The idea

Think of each run as a weighted coin that comes up "failure" with some small probability, which we'll call **p**. We don't know p, and all we know is that in 300 runs we saw no failures.

- If p were large (e.g., 5%), a perfect 300-run record would be almost impossible.
- If p were tiny (e.g., 0.01%), a perfect record would be completely normal.
- Somewhere in between is a value of p where a perfect record would be surprising but still possible.

We find that value and say "the true rate is probably below this," which is all an upper bound really is.

**The one formula you need.** The chance that a single run succeeds is $1 - p$, so if runs are independent, the chance that all $n$ runs succeed is $1 - p$ multiplied by itself $n$ times.

$$\text{chance of a perfect record} = (1 - p)^n$$

## A worked example

Run the cell below, which asks how often 300 runs would come out perfect for a few possible failure rates.

In [ ]:
n = 300
for p in [0.001, 0.005, 0.01, 0.02]:
    chance_all_clean = (1 - p) ** n
    print(f"If the true failure rate were {p:.1%}, a perfect record in {n} runs happens {chance_all_clean:.1%} of the time")

If the true failure rate were 0.1%, a perfect record in 300 runs happens 74.1% of the time
If the true failure rate were 0.5%, a perfect record in 300 runs happens 22.2% of the time
If the true failure rate were 1.0%, a perfect record in 300 runs happens 4.9% of the time
If the true failure rate were 2.0%, a perfect record in 300 runs happens 0.2% of the time


**Read the output.** Even if the monitor missed one attack in 200 (i.e., 0.5%), a perfect 300-run record would still happen about a fifth of the time, and only at around 1% does a perfect record become roughly a 1-in-20 coincidence. So 300 clean runs rule out miss rates above about 1% but can't rule out lower ones, which means "zero" is not on the table.

You can also check this by simulation, which you'll do often in this lab. The next cell plays out 100,000 imaginary 300-run test campaigns in which the true miss rate is 0.5%, and counts how many of them come out perfect.

In [ ]:
rng = np.random.default_rng(1)
p, n, campaigns = 0.005, 300, 100_000
misses = rng.binomial(n, p, size=campaigns)   # misses in each simulated campaign
print(f"Share of simulated campaigns with zero misses: {(misses == 0).mean():.1%}")

Share of simulated campaigns with zero misses: 22.2%


The simulation lands on about 22%, matching the formula, so whenever you're unsure about a formula, try simulating it.

## Part 2. Your turn (the chance of a perfect record)

**Step 1.** Write `prob_zero_failures(p, n)`, which should return the chance of seeing no failures in `n` runs when each run fails with probability `p` (you used the formula in the worked example).

In [ ]:
def prob_zero_failures(p, n):
    """Chance of no failures in n independent runs, each failing with probability p."""
    return (1 - p) ** n

In [ ]:
check("prob_zero_failures", prob_zero_failures,
      [((0.005, 300), 0.2223), ((0.01, 300), 0.04904), ((0.0, 50), 1.0)],
      tol=1e-3, hint="Raise (1 - p) to the power n. In Python, a ** b means a to the power b.")

Correct: prob_zero_failures passed 3 checks.


<details><summary>Hint</summary>

`return (1 - p) ** n`
</details>

## Part 3. Your turn (the upper bound)

Now turn the question around and ask, given that we saw zero failures, **what's the highest failure rate that still makes that reasonably likely?**

The usual standard is 95% confidence, which means we rule out any rate under which our perfect record would be a less-than-5% coincidence. So we look for the p where the chance of a perfect record falls to exactly 5%.

$$(1 - p)^n = 0.05$$

You can solve it for p in two steps.

1. Take the n-th root of both sides to get $1 - p = 0.05^{1/n}$.
2. Rearrange to get $p = 1 - 0.05^{1/n}$.

The 0.05 is just 1 minus the confidence level (i.e., 1 − 0.95), and in Python the n-th root of `x` is `x ** (1 / n)`.

**Step 2.** Write `zero_failure_upper_bound(n, confidence)` using the result of step 2, with `1 - confidence` in place of 0.05.

In [ ]:
def zero_failure_upper_bound(n, confidence=0.95):
    """Highest failure rate still consistent (at this confidence) with zero failures in n runs."""
    return 1 - (1 - confidence) ** (1 / n)

In [ ]:
check("zero_failure_upper_bound", zero_failure_upper_bound,
      [((300,), 0.009937), ((30,), 0.09503), ((300, 0.99), 0.01523)],
      tol=1e-4, hint="1 - (1 - confidence) ** (1 / n). Check your brackets.")

Correct: zero_failure_upper_bound passed 3 checks.


<details><summary>Hint</summary>

`return 1 - (1 - confidence) ** (1 / n)`. The brackets around `1 / n` matter, because without them Python divides after raising to the power.
</details>

So 300 perfect runs support "the miss rate is below about 1%, with 95% confidence."

## Part 4. Your turn (the rule of three)

There's a shortcut that comes in handy in meetings, which is that for anything beyond a few dozen runs the bound is about **3 divided by the number of runs** (e.g., 300 clean runs give about 1%, and 1,000 give about 0.3%).

*Why 3?* The natural log of 0.05 is about −3, and for small p the log of (1 − p) is about −p, so the equation $(1-p)^n = 0.05$ becomes roughly $-p \times n = -3$, which gives $p \approx 3/n$.

**Step 3.** Write `rule_of_three(n)`.

In [ ]:
def rule_of_three(n):
    """Approximate 95% upper bound on a failure rate after zero failures in n runs."""
    return 3 / n

In [ ]:
check("rule_of_three", rule_of_three, [(300, 0.01), (1000, 0.003)], hint="Three divided by n.")

Correct: rule_of_three passed 2 checks.


Now compare the shortcut with the exact bound. From here on the chapters use the toolkit's reference versions (i.e., `labkit`), so you can keep going even if a step above isn't finished.

In [ ]:
from labkit import stats as kit

print(" runs   exact bound   rule of three")
for n in [10, 30, 100, 300, 1000, 3000]:
    print(f"{n:>5}   {kit.zero_failure_upper_bound(n):>10.2%}   {kit.rule_of_three(n):>12.2%}")

 runs   exact bound   rule of three
   10       25.89%         30.00%
   30        9.50%         10.00%
  100        2.95%          3.00%
  300        0.99%          1.00%
 1000        0.30%          0.30%
 3000        0.10%          0.10%


The shortcut overstates a little when there are only a few runs (e.g., 30% against 26% at 10 runs) and is very close from about 30 runs on, so use the exact number in writing and the rule of three in conversation.

## Part 5. Your turn (how many runs would a stronger claim need?)

Suppose the lab wants to say the miss rate is **below 0.1%**. How many clean runs does that take?

One way to think about it is that if the true rate were 0.1%, we'd want enough runs that we'd very likely (i.e., 95% of the time) see at least one failure, because then a perfect record really would tell us something.

- The chance of at least one failure is the opposite of a perfect record, which is $1 - (1-p)^n$.
- We want that to be at least 0.95, which is the same as $(1-p)^n \le 0.05$.
- Taking the log of both sides gives $n \times \ln(1-p) \le \ln(0.05)$.
- $\ln(1-p)$ is negative, so dividing by it flips the inequality and gives $n \ge \ln(0.05) / \ln(1-p)$.
- Runs come in whole numbers, so round up.

**Step 4.** Write `trials_needed(p, confidence)`, using `math.log` for the natural log and `math.ceil` to round up.

In [ ]:
def trials_needed(p, confidence=0.95):
    """Fewest runs that would show at least one failure with this confidence, if the true rate were p."""
    return math.ceil(math.log(1 - confidence) / math.log(1 - p))

In [ ]:
check("trials_needed", trials_needed, [(0.01, 299), (0.001, 2995), (0.05, 59)],
      hint="math.ceil(math.log(1 - confidence) / math.log(1 - p))")

Correct: trials_needed passed 3 checks.


<details><summary>Hint</summary>

`return math.ceil(math.log(1 - confidence) / math.log(1 - p))`. For a quick sanity check, the answer should be close to 3 divided by p.
</details>

## Part 6. Check your understanding

Back to the lab's claim, which was *"The monitor caught all 300 attempts, so the miss rate is zero."*

In [ ]:
n_attempts = 300
print(f"95% upper bound on the miss rate: {kit.zero_failure_upper_bound(n_attempts):.2%}")
print(f"Clean attempts needed to claim below 0.1%: {kit.trials_needed(0.001):,}")

95% upper bound on the miss rate: 0.99%
Clean attempts needed to claim below 0.1%: 2,995


### What the bound does not cover

The math assumes two things, and you should keep both in view, because this is where safety claims usually fall down.

1. **The runs are independent.** If the red team reused one attack style 300 times, you effectively have far fewer independent tries than the count suggests. A later chapter shows how to account for that.
2. **The attacks you tested are like the ones that matter.** A perfect record against weak attacks says nothing about strong ones, so a safety case needs a strong red team rather than just many runs. That point is easy to skim past, and it's the heart of the safety-case exercise in the next chapter.

### Your turn to explain it in your own words

In the cell below, explain in two or three sentences what those 300 clean runs do and don't show, as if to a smart friend who hasn't read this chapter (i.e., say what the data supports, what it doesn't and what would strengthen it). Double-click the cell to edit it.

**Model answer.** The monitor caught all 300 red-team attempts, which supports a miss rate below about 1% (i.e., the 95% upper bound is 0.99%) rather than a miss rate of zero, and that bound only covers attacks like the ones tested. Supporting "below 0.1%" would take about 3,000 clean attempts, though a stronger, more varied red team matters at least as much as more runs.

## Recap

- Zero failures in n runs doesn't mean the rate is zero, only that it's probably below $1 - 0.05^{1/n}$, which is roughly 3/n.
- To support a claim that a rate is below p, you need about $\ln(0.05)/\ln(1-p)$ clean runs (i.e., roughly 3/p).
- Always say what kind of runs the bound covers, since it tells you nothing about the kinds you didn't test.

## Self-check

<details><summary>1. A lab ran 1,000 clean tests. What can it claim?</summary>

The failure rate is probably below about 0.3% (exactly 0.30%) for the kinds of tests it ran.
</details>

<details><summary>2. Why can't 300 clean runs show a rate is zero?</summary>

Small but real rates (e.g., 0.5%) still produce a perfect 300-run record fairly often, about 22% of the time.
</details>

<details><summary>3. Roughly how many clean runs would support "below 1 in 10,000"?</summary>

About 3 / 0.0001 = 30,000 (exactly 29,956).
</details>

**Next.** Do the Chapter 1 exercise, the threat model, in `exercises/ch01_threat_model`, and when you list the tests you'd run first, use this chapter to say how many runs each one would need.